# Persistent Colab training: 3B CoT teacher → looped 1B student

**First session:** run cells **1 → 2 → 3 → 4 → 5**. After training, run **6 → 7**.
**New session:** run **1 → 2 → 3 → 5**. Cell 4 is only needed to prepare or repair data.

Select a **high-RAM GPU runtime** before starting. This full-finetuning recipe
requires native BF16, at least **35 GiB GPU memory**, and at least **30 GiB host RAM**.
These are conservative preflight gates, not measured capacity guarantees. T4 and
24 GiB L4 runtimes are rejected. The default is **microbatch 2 / accumulation 64 = 128**.

Add a private **HF_TOKEN** in Colab's Secrets panel and enable notebook access.
The token's Hugging Face account needs access to the Llama 3.2 1B and 3B models.
Only use your own dedicated Drive folder; allow approximately **60 GiB free** for
model caches, checkpoints, best weights and exports. Check your school's storage quota.

**Persistence boundary:** a checkpoint is committed after an epoch's training and
validation finish. A disconnect repeats the unfinished epoch from the last
verified completed checkpoint. There is no mid-epoch recovery. Long fully latent
epochs may exceed Colab's session lifetime; GPU availability and runtime duration
are not guaranteed. Drive completion markers detect incomplete files but cannot
guarantee that Drive has finished syncing to its servers. Run one session per RUN_ID.

Published CoT 1B/3B checkpoints are inexpensive-to-set-up reference baselines for
evaluation. They have not received this run's continued training and therefore
do not isolate the distillation effect. No additional baseline training runs are launched.

## 1 · Settings and public repository

Edit these settings before the first run. Keep **DRIVE_ROOT, RUN_ID, batch profile,
epochs and training/validation limits identical** when resuming. Use a new RUN_ID
for a different experiment. `None` means the full original split.

Cell 3 restores an existing run's recorded code commit and package lock automatically.
The notebook's Python kernel stays separate from the training virtualenv, so installing
dependencies does not require a kernel restart.

In [ ]:
from pathlib import Path
import json, os, signal, subprocess, sys, time

REPOSITORY = "https://github.com/sttruji/lotus-smaller-student.git"
REPO_REF = "main"
DRIVE_ROOT = "/content/drive/MyDrive/lotus-research"  # Your dedicated persistent folder
RUN_ID = "gsm-distill-b128-mb2-seed0"
MICROBATCH, ACCUMULATION = 2, 64  # Alternative, after profiling: 8, 16
EPOCHS = 10
TRAIN_MAX_EXAMPLES = None  # Optional small pipeline pilot; use a NEW RUN_ID
VAL_MAX_EXAMPLES = None
EVAL_MAX_EXAMPLES = None   # None = all 1,319 test examples; a limit is labeled a SUBSET
EVAL_MAX_NEW_TOKENS = 512   # Same output cap for student and all baselines
EVALUATE_BASELINES = True  # Evaluate published CoT 1B and 3B; no extra training
BASELINE_METRICS = []      # Optional absolute Drive paths to comparable schema-2 reports

REPO_DIR = Path("/content/lotus-smaller-student")
VENV_DIR = Path("/content/lotus-training-venv")
PYTHON = VENV_DIR / "bin/python"
SESSION_FILE = Path("/content/lotus-colab-session.json")
if not Path("/content").is_dir():
    raise RuntimeError("Open this notebook in Google Colab")
if REPO_DIR.exists():
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if origin != REPOSITORY:
        raise RuntimeError("Existing /content checkout has a different origin; choose another REPO_DIR")
    dirty = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True)
    if dirty.strip():
        raise RuntimeError("Tracked code was edited. Preserve your edits before replacing the checkout")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
print("Repository:", REPO_DIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())
print("Persistent run:", Path(DRIVE_ROOT) / "runs" / RUN_ID)

## 2 · Install dependencies in an isolated runtime environment

Installs CUDA PyTorch 2.7.1 and the repo's Colab requirements. Existing Colab
kernel packages are untouched. Installation is repeatable; pip reuses installed
packages within the same VM. A new VM must rebuild this ephemeral environment.
The environment is created without ensurepip; Colab's existing pip bootstraps
pip into the target environment, including after a partially failed creation.

In [ ]:
import venv

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("This pinned PyTorch environment supports Python 3.10–3.13; choose a compatible Colab runtime")
# Avoid ensurepip, which failed in the reported Colab Python 3.13 runtime.
# Complete a partially created environment even if bin/python already exists.
venv.EnvBuilder(with_pip=False, clear=False).create(VENV_DIR)
subprocess.run([sys.executable, "-m", "pip", "--python", str(PYTHON),
                "install", "--upgrade", "pip"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "torch==2.7.1",
                "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-colab.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
subprocess.run([str(PYTHON), "-c",
    "import torch, datasets, transformers; from transformers import LlamaForCausalLM; "
    "print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'transformers', transformers.__version__)"], check=True)
print("Dependencies ready; continue to cell 3. No kernel restart is needed.")

## 3 · Mount Drive, authenticate, and restore the recorded run environment

Google will request Drive authorization. HF_TOKEN comes from Colab Secrets or a
hidden prompt, stays in memory, and is inherited by training subprocesses.
Data, model downloads, checkpoints, logs, exports and results persist in DRIVE_ROOT.
Tokens and the Python environment are not saved to Drive.

In [ ]:
from google.colab import drive, userdata
from getpass import getpass

drive.mount("/content/drive")
drive_root = Path(DRIVE_ROOT)
if not os.path.ismount("/content/drive") or not drive_root.resolve().is_relative_to(Path("/content/drive/MyDrive").resolve()):
    raise RuntimeError("DRIVE_ROOT must be inside the mounted MyDrive")
drive_root.mkdir(parents=True, exist_ok=True)
probe = drive_root / ".lotus-write-probe"
probe.write_text("Drive write access confirmed")
probe.unlink()
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = getpass("Hugging Face read token (hidden; not saved to Drive): ")
if not token or not token.strip():
    raise RuntimeError("Set HF_TOKEN with access to Llama 3.2 1B and 3B, then rerun cell 3")
os.environ["HF_TOKEN"] = token.strip()
del token
os.environ["HF_HOME"] = "/content/lotus-hf-home"
os.environ["HF_HUB_CACHE"] = str(drive_root / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = "/content/lotus-datasets-cache"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

run_dir = drive_root / "runs" / RUN_ID
plan_file = run_dir / "run_plan.json"
if plan_file.exists():
    saved_plan = json.loads(plan_file.read_text())
    if saved_plan["identity"]["python"] != list(sys.version_info[:2]):
        raise RuntimeError("This run used a different Python version. Select the matching Colab runtime before resuming")
    saved_commit = saved_plan["identity"]["git_commit"]
    subprocess.run(["git", "fetch", "--depth", "1", "origin", saved_commit], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", saved_commit], cwd=REPO_DIR, check=True)
    frozen = run_dir / "requirements.freeze.txt"
    if not frozen.is_file():
        raise RuntimeError("Saved run is missing its package lock; inspect the persistent run directory")
    subprocess.run([str(PYTHON), "-m", "pip", "install", "--extra-index-url",
                    "https://download.pytorch.org/whl/cu126", "-r", str(frozen)], check=True)
    print("Restored recorded commit and packages:", saved_commit)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
SESSION_FILE.write_text(json.dumps({
    "drive_root": str(drive_root), "run_id": RUN_ID,
    "microbatch": MICROBATCH, "accumulation": ACCUMULATION, "epochs": EPOCHS,
    "train_max_examples": TRAIN_MAX_EXAMPLES, "val_max_examples": VAL_MAX_EXAMPLES,
    "eval_max_examples": EVAL_MAX_EXAMPLES, "eval_max_new_tokens": EVAL_MAX_NEW_TOKENS,
    "evaluate_baselines": EVALUATE_BASELINES, "baseline_metrics": BASELINE_METRICS,
}, indent=2))

def run_action(action):
    command = [str(PYTHON), str(REPO_DIR / "scripts/colab_runner.py"), action,
               "--session", str(SESSION_FILE)]
    process = subprocess.Popen(command, cwd=REPO_DIR, env=os.environ.copy(),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, errors="replace", start_new_session=True)
    try:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
        code = process.wait()
        if code:
            raise RuntimeError(f"{action} exited {code}; inspect the output and persistent logs")
    finally:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGINT)  # Helper closes its torchrun group in finally
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                subprocess.run(["pkill", "-TERM", "-P", str(process.pid)], check=False)
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        process.stdout.close()

print("Drive and authentication ready. Dataset:", drive_root / "datasets/gsm8k-icot-e06a32ee")
print("Run folder:", run_dir)

## 4 · One-time dataset download to Drive

Downloads the **pinned original augmented GSM8K train/valid/test splits**;
verifies raw source SHA-256, expected counts and prepared JSON hashes. Re-running
this cell verifies existing data and skips the download. Training never silently
downloads or replaces a missing dataset. This cell can also repair incomplete data.

In [ ]:
run_action("prepare-data")

## 5 · Train or automatically resume

Run after **1, 2, 3** on every new session. The original dataset is reused from
Drive. The recipe, code, dependencies and model revisions are locked on first launch;
incompatible settings fail with instructions to choose a new RUN_ID.

Writes model, projection, optimizer, scheduler, RNG and curriculum progress each
completed epoch to **DRIVE_ROOT/runs/RUN_ID/checkpoint_N/**. Completion markers and
SHA-256 checks prevent loading a partially written directory. Keeps the latest two
completed epoch checkpoints, plus separately verified best-model generations.
Corrupt newest epochs fall back to an older verified epoch. If all completed
checkpoints are corrupt, it stops instead of silently restarting training.

Stop the cell normally to terminate its training subprocesses. A hard runtime loss
can only recover the latest completed epoch. Full stage-6 training memory and speed
have not been measured; the notebook does not reduce model size, precision or batch
silently to fit a smaller GPU. If training is already complete, this cell reports it
without starting another training job.

In [ ]:
run_action("train")

## 6 · Evaluate the completed student and reference baselines

Requires all requested training epochs and a verified best checkpoint from the fully
latent stage. Exports student-only HF weights, then evaluates the student and (by
default) the published CoT **1B** and **3B** models sequentially on the same persisted
test population with the same output cap. Baselines are pinned to the exact checkpoints
used to initialize this run. No teacher or training projections are loaded for student inference.

Saves predictions, complete schema-2 metrics and a comparison index in Drive.
Rerunning skips completed, checksummed evaluations with identical model, data, settings,
GPU and code. An interrupted individual model evaluation restarts that model.
Set EVALUATE_BASELINES=False in cell 1 to defer their compute, rerun cell 3, then this cell.

In [ ]:
run_action("evaluate")

## 7 · Visualize test results against baselines

Displays accuracy, output truncation, full-generation latency and curriculum
validation history. Also saves PNG plots and CSV tables to **RUN_ID/results/**.
Baseline imports must use the same ordered population, output cap, decoding,
dtype and evaluation batch size. Failed or incomplete evaluations are rejected.
Latency comparisons are omitted when GPU or library versions differ.

Published CoT checkpoints provide reference comparisons; a controlled KD ablation
requires separate matched continued-training runs. Published paper headline numbers
are not mixed into these measured plots.

In [ ]:
run_action("plot")
from IPython.display import display, Image

results_dir = Path(DRIVE_ROOT) / "runs" / RUN_ID / "results"
for filename in ("baseline_comparison.png", "training_history.png"):
    if (results_dir / filename).is_file():
        display(Image(filename=str(results_dir / filename)))
print("CSV tables and plots:", results_dir)

### References and limitations

- [Repository and research docs](https://github.com/sttruji/lotus-smaller-student)
- [Google Colab FAQ: runtime limits and Drive I/O](https://research.google.com/colaboratory/faq.html)
- [Hugging Face authentication and Colab Secrets](https://huggingface.co/docs/huggingface_hub/quick-start#authentication)
- [Official PyTorch 2.7.1 CUDA wheel instructions](https://pytorch.org/get-started/previous-versions/#v271)

This notebook has not been run on a school Colab account or GPU. School policies,
available GPUs, Drive permissions/quota and Colab service limits still apply.
Only mount Drive and load checkpoints that you trust; optimizer/RNG restoration
requires Python pickle deserialization of this run's own training state.